In [ ]:
# 1- Initial Imports
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms

In [ ]:
# 2- Dataset Initialization Phase

import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import torch
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import random
from natsort import natsorted
import platform

class SegmentationDataset(Dataset):
    def __init__(self, dataset_type, num_classes=None, augment=False):
        
        base_dir = r"C:\Users\esmat\Desktop\Master Dataset Alpha - Class Indices"

        if dataset_type not in ["train", "val", "test"]:
            raise ValueError("dataset_type must be 'train', 'val', or 'test'")

        self.image_dir = os.path.join(base_dir, dataset_type, "image")
        self.mask_dir  = os.path.join(base_dir, dataset_type, "mask")

        self.image_filenames = natsorted(os.listdir(self.image_dir))
        self.mask_filenames  = natsorted(os.listdir(self.mask_dir))

        # safety check
        if len(self.image_filenames) != len(self.mask_filenames):
            raise ValueError("Number of images and masks does not match")

        self.num_classes = num_classes
        self.augment = augment

        # ImageNet normalization (correct for pretrained ResNet backbone)
        dataset_mean = [0.485, 0.456, 0.406]
        dataset_std  = [0.229, 0.224, 0.225]

        self.transform_image = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=dataset_mean, std=dataset_std)
        ])

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):

        filename = self.image_filenames[idx]

        img_path  = os.path.join(self.image_dir, self.image_filenames[idx])
        mask_path = os.path.join(self.mask_dir,  self.mask_filenames[idx])

        # Load image (RGB)
        image = Image.open(img_path).convert("RGB")

        # Load mask (single-band TIFF with class indices)
        mask = Image.open(mask_path)
        mask_np = np.array(mask)

        # Ensure mask is single band
        if mask_np.ndim != 2:
            raise ValueError(f"Mask must be single band. Got shape: {mask_np.shape}")

        # Data augmentation (must apply same transform to image and mask)
        if self.augment:
            if random.random() > 0.5:
                image = TF.hflip(image)
                mask_np = np.fliplr(mask_np)

            if random.random() > 0.5:
                image = TF.vflip(image)
                mask_np = np.flipud(mask_np)
        # Transform image
        image = self.transform_image(image)

        # Convert mask to tensor (class indices)
        mask_tensor = torch.from_numpy(mask_np.copy()).long()

        # Safety check for class ID range
        if self.num_classes is not None:
            if mask_tensor.max() >= self.num_classes or mask_tensor.min() < 0:
                raise ValueError(
                    f"Mask contains invalid class IDs. "
                    f"Found range [{mask_tensor.min()}, {mask_tensor.max()}], "
                    f"expected [0, {self.num_classes-1}]"
                )
        return image, mask_tensor, filename

# Safe entry point for Windows multiprocessing
if __name__ == "__main__":
    num_classes = 3
    dataset = SegmentationDataset(
        dataset_type="train",
        num_classes=num_classes,
        augment=True
    )
    num_workers = 0 if platform.system() == "Windows" else 4
    train_loader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=True,
        num_workers=num_workers
    )

    # Test one batch
    images, masks, filenames = next(iter(train_loader))
    print("Image batch shape:", images.shape)   # [B, 3, H, W]
    print("Mask batch shape:", masks.shape)     # [B, H, W]
    print("Classes in batch:", torch.unique(masks))

In [ ]:
# 3- Preparing the datasets and dataloaders
from torch.utils.data import DataLoader
import platform

if __name__ == "__main__":
    train_dataset = SegmentationDataset(dataset_type="train", num_classes=num_classes, augment=True)
    val_dataset = SegmentationDataset(dataset_type="val", num_classes=num_classes, augment=False)
    test_dataset = SegmentationDataset(dataset_type="test", num_classes=num_classes, augment=False)
    num_workers = 0 if platform.system() == "Windows" else 4

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=num_workers)

In [ ]:
# 4- DeepLabV3 segmentation model with a custom backbone

import torch
import torch.nn as nn
from torchvision.models.segmentation import DeepLabV3
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
from torchvision.models import resnet101

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 3


class ResNet101BackboneWrapper(nn.Module):
    def __init__(self):
        super().__init__()

        base_model = resnet101(
            weights="IMAGENET1K_V1",
            replace_stride_with_dilation=[False, True, True]  # important for DeepLab
        )

        # Remove avgpool and fc layers
        self.backbone = nn.Sequential(*list(base_model.children())[:-2])

    def forward(self, x):
        x = self.backbone(x)
        return {"out": x}


# Define model
backbone = ResNet101BackboneWrapper()

model = DeepLabV3(
    backbone=backbone,
    classifier=DeepLabHead(2048, num_classes)
)

model = model.to(device)

print(model)

In [ ]:
# 5- Setup of loss function, optimizer, and learning rate scheduler
from torch import nn, optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)  # Initial learning rate

# StepLR scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)  

In [ ]:
# 6- Setup of Training Function (FIXED for Dataset returning image, mask, filename)
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

def train_model(train_loader, val_loader, model, criterion, optimizer,
                scheduler=None, num_epochs=1, patience=20, ignore_class=None):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    num_classes = 3

    # ==================== IOU FUNCTION ====================
    def compute_iou(preds, labels, num_classes, ignore_class=ignore_class):
        preds = preds.view(-1)
        labels = labels.view(-1)
        ious = []
        for cls in range(num_classes):
            if cls == ignore_class:
                continue
            pred_inds = preds == cls
            label_inds = labels == cls
            intersection = (pred_inds & label_inds).sum().item()
            union = pred_inds.sum().item() + label_inds.sum().item() - intersection
            if union == 0:
                ious.append(float("nan"))
            else:
                ious.append(intersection / union)
        return ious

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        # ==================== TRAIN ====================
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        for images, labels, _ in train_loader:  # <-- unpack 3 values
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)["out"]
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.numel()

        train_accuracy = 100 * correct_train / total_train
        avg_train_loss = train_loss / len(train_loader)

        # ==================== VALIDATION ====================
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        val_ious = []
        all_val_preds = []
        all_val_labels = []

        with torch.no_grad():
            for images, labels, _ in val_loader:  # <-- unpack 3 values
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)["out"]
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.numel()

                val_ious.append(
                    compute_iou(preds, labels, num_classes, ignore_class=ignore_class)
                )

                all_val_preds.append(preds.cpu().numpy().flatten())
                all_val_labels.append(labels.cpu().numpy().flatten())

        val_accuracy = 100 * correct_val / total_val
        avg_val_loss = val_loss / len(val_loader)

        # ==================== METRICS ====================
        val_ious = np.array(val_ious)
        mean_iou_per_class = np.nanmean(val_ious, axis=0)
        mean_iou = np.nanmean(mean_iou_per_class)

        all_val_preds = np.concatenate(all_val_preds)
        all_val_labels = np.concatenate(all_val_labels)

        conf_matrix = confusion_matrix(all_val_labels, all_val_preds)
        f1 = f1_score(all_val_labels, all_val_preds, average=None)
        precision = precision_score(all_val_labels, all_val_preds, average=None, zero_division=0)
        recall = recall_score(all_val_labels, all_val_preds, average=None, zero_division=0)

        # ==================== PRINT ====================
        print(
            f"\nEpoch [{epoch+1}/{num_epochs}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.2f}% | "
            f"Val mIoU (all classes): {mean_iou:.4f}"
        )

        print("IoU per class:")
        for i, iou in enumerate(mean_iou_per_class):
            print(f"  Class {i}: {iou:.4f}")

        print("F1 score per class:")
        for i, score in enumerate(f1):
            print(f"  Class {i}: {score:.4f}")

        print("Precision per class:")
        for i, score in enumerate(precision):
            print(f"  Class {i}: {score:.4f}")

        print("Recall per class:")
        for i, score in enumerate(recall):
            print(f"  Class {i}: {score:.4f}")

        print("Confusion Matrix:")
        print(conf_matrix)

        # ==================== SCHEDULER, CHECKPOINT & EARLY STOP ====================
        if scheduler is not None:
            scheduler.step()

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": best_val_loss,
            }, "best_checkpoint.pth")

            print(" Checkpoint saved (best validation loss)")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    return model

In [ ]:
# 7- testing, evaluation, and saving predictions

import os
import numpy as np
import torch
import rasterio
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

# ================= SETTINGS =================
output_dir = r"C:\Users\esmat\Desktop\Master Dataset Alpha - Class Indices\test\predictions"
os.makedirs(output_dir, exist_ok=True)

num_classes = 3
ignore_class = None

# ================= IOU FUNCTION =================
def compute_iou(all_preds, all_labels, num_classes, ignore_class=None):
    ious = []
    for cls in range(num_classes):
        if ignore_class is not None and cls == ignore_class:
            continue
        intersection = np.logical_and(all_preds == cls, all_labels == cls).sum()
        union = np.logical_or(all_preds == cls, all_labels == cls).sum()
        if union == 0:
            ious.append(np.nan)
        else:
            ious.append(intersection / union)
    return np.array(ious)


# ================= TEST FUNCTION =================
def test_model(test_loader, model, criterion):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    running_loss = 0.0
    total_correct = 0
    total_pixels = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels, filenames in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)["out"]
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)

            # ================= SAVE PREDICTIONS =================
            for i in range(predicted.shape[0]):
                pred_mask = predicted[i].cpu().numpy().astype(np.uint8)
                base_filename = os.path.splitext(filenames[i])[0]
                tif_path = os.path.join(output_dir, base_filename + ".tif")
                with rasterio.open(
                    tif_path,
                    "w",
                    driver="GTiff",
                    height=pred_mask.shape[0],
                    width=pred_mask.shape[1],
                    count=1,
                    dtype="uint8"
                ) as dst:
                    dst.write(pred_mask, 1)

            # ================= ACCURACY (GLOBAL CORRECT) =================
            correct = (predicted == labels).sum().item()
            total = labels.numel()
            total_correct += correct
            total_pixels += total
            all_preds.append(predicted.cpu().numpy().flatten())
            all_labels.append(labels.cpu().numpy().flatten())

    # ================= FINAL METRICS =================
    avg_loss = running_loss / len(test_loader)
    avg_accuracy = total_correct / total_pixels
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    conf_matrix = confusion_matrix(all_labels, all_preds, labels=[0,1,2])
    f1 = f1_score(all_labels, all_preds, labels=[0,1,2], average=None)
    precision = precision_score(all_labels, all_preds, labels=[0,1,2], average=None, zero_division=0)
    recall = recall_score(all_labels, all_preds, labels=[0,1,2], average=None, zero_division=0)
    ious = compute_iou(all_preds, all_labels, num_classes, ignore_class)
    miou = np.nanmean(ious)

    producer_accuracy = np.diag(conf_matrix) / np.sum(conf_matrix, axis=1)
    user_accuracy = np.diag(conf_matrix) / np.sum(conf_matrix, axis=0)

    # ================= PRINT RESULTS =================
    print(f"\nTest Loss: {avg_loss:.4f}")
    print(f"Global Pixel Accuracy: {avg_accuracy:.4f}")
    print(f"Mean IoU: {miou:.4f}")

    print("\nIoU per class:")
    for i, iou in enumerate(ious):
        print(f"  Class {i}: {iou:.4f}")

    print("\nF1 Score per class:")
    for i, score in enumerate(f1):
        print(f"  Class {i}: {score:.4f}")

    print("\nPrecision per class:")
    for i, score in enumerate(precision):
        print(f"  Class {i}: {score:.4f}")

    print("\nRecall per class:")
    for i, score in enumerate(recall):
        print(f"  Class {i}: {score:.4f}")

    print("\nConfusion Matrix:")
    print(conf_matrix)

    print("\nProducer Accuracy per class:")
    print(producer_accuracy)

    print("\nUser Accuracy per class:")
    print(user_accuracy)

    return {
        "loss": avg_loss,
        "accuracy": avg_accuracy,
        "iou": ious,
        "miou": miou,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "confusion_matrix": conf_matrix,
        "producer_accuracy": producer_accuracy,
        "user_accuracy": user_accuracy
    }

In [ ]:
# 8- Training Phase

if __name__ == "__main__":
    num_epochs = 50
    patience = 20
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # ================= TRAIN =================
    trained_model = train_model(
        train_loader=train_loader,
        val_loader=val_loader,
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=num_epochs,
        patience=patience
    )
    
    # ================= SAVE FINAL MODEL =================
    model_save_path = r"C:/My Files/Semester4TUM/Developed Dataset/Prepared Dataset for Thesis - fixed class imbalance/7indice_model_resnet101.pth"
    torch.save(trained_model.state_dict(), model_save_path)
    print(f"Model saved to {model_save_path}")
    
    # ================= LOAD BEST CHECKPOINT =================
    checkpoint = torch.load("best_checkpoint.pth", map_location=device)
    trained_model.load_state_dict(checkpoint["model_state_dict"])
    print("Loaded best validation checkpoint")
    
# ================= TEST =================
    test_metrics = test_model(
        test_loader,
        trained_model,
        criterion
     )
print("\nTest Accuracy (overall pixel accuracy):", test_metrics["accuracy"])

In [ ]:
# ================= TEST =================
test_metrics = test_model(
        test_loader,
        trained_model,
        criterion
     )
print("\nTest Accuracy (overall pixel accuracy):", test_metrics["accuracy"])

In [ ]:
#9 Loading The Model Back

backbone = ResNet101BackboneWrapper()

model = DeepLabV3(
    backbone=backbone,
    classifier=DeepLabHead(2048, num_classes)
)

model.load_state_dict(torch.load(model_save_path))

model = model.to(device)

model.eval()  # set to evaluation mode

In [ ]:
# 10- Visual test of prediction on test masks

import os
import torch
from PIL import Image
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import rasterio

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
img_dir = r"C:\Users\esmat\Desktop\Master Dataset Alpha - Class Indices\test\image"
mask_dir = r"C:\Users\esmat\Desktop\Master Dataset Alpha - Class Indices\test\test"
pred_save_dir = r"C:\Users\esmat\Desktop\Master Dataset Alpha - Class Indices\test\predictions"

os.makedirs(pred_save_dir, exist_ok=True)

# Dataset normalization
dataset_mean = [0.485, 0.456, 0.406]  
dataset_std  = [0.229, 0.224, 0.225] 

# Transform for model input
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=dataset_mean, std=dataset_std)
])

# Colormap: 0=Formal (Black), 1=Other (Blue), 2=Informal (Red)
colormap = np.array([
    [74, 100, 145],       # Class 0
    [224, 224, 224],     # Class 1
    [217, 95, 14],     # Class 2
], dtype=np.uint8)

sorted_filenames = sorted([f for f in os.listdir(img_dir) if f.endswith(".tif")])

for idx, filename in enumerate(sorted_filenames):
    # a - Load Original Image
    img_path = os.path.join(img_dir, filename)
    raw_img = Image.open(img_path).convert("RGB")    #check this????
    
    # b - Prepare Input and Predict
    input_tensor = transform(raw_img).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(input_tensor)['out']
        predicted_indices = torch.argmax(outputs, dim=1).squeeze(0).cpu().numpy()

    # c - Save predicted mask with consistent naming
    base_name = os.path.splitext(filename)[0]
    pred_save_path = os.path.join(pred_save_dir, f"{idx:03d}_{base_name}_pred.tif")  # e.g., 000_image1_pred.tif
    height, width = predicted_indices.shape
    with rasterio.open(
        pred_save_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=1,
        dtype=predicted_indices.dtype
    ) as dst:
        dst.write(predicted_indices, 1)

    # d - Load Ground Truth Mask
    mask_path = os.path.join(mask_dir, filename)
    if os.path.exists(mask_path):
        with rasterio.open(mask_path) as src:
            gt_indices = src.read(1)
    else:
        gt_indices = np.zeros_like(predicted_indices)

    gt_indices = np.clip(gt_indices, 0, len(colormap)-1).astype(np.int64)
    pred_indices_clean = np.clip(predicted_indices, 0, len(colormap)-1).astype(np.int64)

    gt_colored = colormap[gt_indices].astype(np.uint8)
    pred_colored = colormap[pred_indices_clean].astype(np.uint8)

    # e - Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(raw_img)
    axes[0].axis("off")

    axes[1].imshow(gt_colored)
    axes[1].axis("off")

    axes[2].imshow(pred_colored)
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

print(f"Inference complete. Results saved to: {pred_save_dir}")

In [ ]:
# 11- works prepping the input image for citywide inference

import rasterio
import numpy as np
from PIL import Image
import torchvision.transforms as T
import torch

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Patch size
patch_height = 224
patch_width = 224

# Stride
stride_y = patch_height // 2  #50% overlap
stride_x = patch_width // 2

# Batch Size
batch_size = 8

# Path to large TIFF image
tiff_path = r"C:\My Files\Semester4TUM\2017 Image Resampled\R3C6.tif"

# Load full resolution image
with rasterio.open(tiff_path) as src:
    img_np = src.read()  # shape: (bands, height, width)
    img_np = np.transpose(img_np, (1, 2, 0))  # (H, W, C)
    img_np = np.clip(img_np, 0, 255).astype(np.uint8)

# Transform
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

H, W, C = img_np.shape

n_patches_y = (H - patch_height) // stride_y + 1
n_patches_x = (W - patch_width) // stride_x + 1


num_classes = 3 #preparing empty arrays for accumulating predictions
output_probs = np.zeros((num_classes, H, W), dtype=np.float32)
count_mask = np.zeros((H, W), dtype=np.float32)

model.eval()

with torch.no_grad():   #sliidng 
    patches = []
    coords = []
    for i in range(n_patches_y):
        for j in range(n_patches_x):
            y = i * stride_y
            x = j * stride_x

            patch = img_np[y:y+patch_height, x:x+patch_width, :]
            patch_pil = Image.fromarray(patch)
            input_tensor = transform(patch_pil)

            patches.append(input_tensor)
            coords.append((y, x))

            # Process batch of the patches
            if len(patches) == batch_size:
                batch_tensor = torch.stack(patches).to(device)  # (batch, C, H, W)
                outputs = model(batch_tensor)['out'] 
                outputs_np = outputs.cpu().numpy()

                for idx, (yy, xx) in enumerate(coords):
                    output_probs[:, yy:yy+patch_height, xx:xx+patch_width] += outputs_np[idx]
                    count_mask[yy:yy+patch_height, xx:xx+patch_width] += 1

                patches = []
                coords = []

    # Process remaining patches
    if len(patches) > 0:
        batch_tensor = torch.stack(patches).to(device)
        outputs = model(batch_tensor)['out']
        outputs_np = outputs.cpu().numpy()

        for idx, (yy, xx) in enumerate(coords):    #stiching back the patches
            output_probs[:, yy:yy+patch_height, xx:xx+patch_width] += outputs_np[idx]
            count_mask[yy:yy+patch_height, xx:xx+patch_width] += 1

# Average overlapping predictions (normalizing)
output_probs /= count_mask[np.newaxis, :, :]

# Final predicted big mask (class with highest probability)
predicted_mask = np.argmax(output_probs, axis=0)

print("Prediction done. Final mask shape:", predicted_mask.shape)

In [ ]:
#12  works batch inference to numpy.py
with torch.no_grad():
    outputs = model(batch_tensor)['out']  # outputs shape: (batch_size, num_classes, H, W)

# Get predicted classes per pixel for all patches in batch
predicted_patches = torch.argmax(outputs, dim=1)  # shape: (batch_size, H, W)

# Convert to numpy array
predicted_patches_np = predicted_patches.cpu().numpy()

In [ ]:
#13 works Full mask visualization   
import numpy as np
import matplotlib.pyplot as plt

# Define the colormap array, index = class id
colormap = np.array([
    [74, 100, 145],       # Class 0 Formal
    [224, 224, 224],     # Class 1 Background
    [217, 95, 14],     # Class 2 Informal
], dtype=np.uint8)

# Map predicted_mask class IDs to colors
color_mask = colormap[predicted_mask]

plt.figure(figsize=(8, 8))
plt.imshow(color_mask)
plt.title("Predicted Mask Colored")
plt.axis('off')
plt.show()


In [ ]:
#14 Save the citywide predicted mask in a singleband format
import rasterio
import numpy as np

# Check which classes are present
unique_classes = np.unique(predicted_mask)
print("Classes present in predicted mask:", unique_classes)

# Open original raster to get metadata
with rasterio.open(tiff_path) as src:
    meta = src.meta.copy()

# Update metadata for single-band
meta.update({
    "count": 1,          # single band
    "dtype": "uint8"     # class indices fit in uint8
})

# Save predicted mask as single-band GeoTIFF
output_tif = r"C:\My Files\Semester4TUM\R3C6_Predict.tif"
with rasterio.open(output_tif, "w", **meta) as dst:
    dst.write(predicted_mask.astype(np.uint8), 1)  # band 1

print("Single-band GeoTIFF with class indices saved:", output_tif)